# 02. Image Multi-Head Model

이 노트북은 얼굴 사진 한 장을 보고 여러 피부 점수를 동시에 예측하는 모델을 학습합니다.

비유하면:

- 사진 한 장은 시험지 한 장
- 백본 모델은 시험지를 읽는 선생님
- 멀티헤드는 국어/영어/수학처럼 여러 과목을 동시에 채점하는 채점창


In [ ]:
!pip install -q -r requirements_colab.txt


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
from pathlib import Path

import torch
from torch.optim import AdamW
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from src.skin_coach.config import DEFAULT_IMAGE_TARGETS
from src.skin_coach.data import ImageMultiTaskDataset
from src.skin_coach.models import ImageMultiHeadModel
from src.skin_coach.utils import load_checkpoint, masked_mse_loss, save_checkpoint, seed_everything

seed_everything(42)
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
device


In [ ]:
PROJECT_ROOT = Path("/content/2026_DNN")
DATA_ROOT = PROJECT_ROOT / "processed"
IMAGE_ROOT = Path("/content/data/images")
DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/2026_DNN/checkpoints/image_model")
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COLUMNS = DEFAULT_IMAGE_TARGETS
BATCH_SIZE = 16
IMAGE_SIZE = 320
EPOCHS = 10
LR = 3e-4
OUTPUT_DIR = DRIVE_OUTPUT_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 이어 학습하려면 아래 경로를 last.pt 또는 best.pt로 바꾸면 됩니다.
RESUME_FROM = OUTPUT_DIR / "last.pt"
RESUME_FROM = RESUME_FROM if RESUME_FROM.exists() else None


In [ ]:
# 데이터셋을 불러옵니다.
# 이 단계는 시험지를 반별로 train/val로 나눠 책상에 올려두는 과정과 같습니다.
train_dataset = ImageMultiTaskDataset(
    csv_path=str(DATA_ROOT / "image_labels.csv"),
    image_root=str(IMAGE_ROOT),
    target_columns=TARGET_COLUMNS,
    split="train",
    image_size=IMAGE_SIZE,
    train=True,
)
val_dataset = ImageMultiTaskDataset(
    csv_path=str(DATA_ROOT / "image_labels.csv"),
    image_root=str(IMAGE_ROOT),
    target_columns=TARGET_COLUMNS,
    split="val",
    image_size=IMAGE_SIZE,
    train=False,
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

sample = train_dataset[0]
print("image shape:", sample["image"].shape)
print("target shape:", sample["targets"].shape)


In [ ]:
model = ImageMultiHeadModel(
    target_columns=TARGET_COLUMNS,
    backbone_name="efficientnet_b3",
).to(device)
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

best_val = float("inf")
start_epoch = 1
if RESUME_FROM is not None:
    checkpoint = load_checkpoint(str(RESUME_FROM), model=model, optimizer=optimizer, map_location=device)
    start_epoch = int(checkpoint.get("epoch", 0)) + 1
    best_val = float(checkpoint.get("extra_state", {}).get("best_val_loss", checkpoint.get("metrics", {}).get("val_loss", float("inf"))))
    print("Resume from:", RESUME_FROM, "start_epoch:", start_epoch)


In [ ]:
def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)

    total_loss = 0.0
    total_steps = 0

    for batch in tqdm(loader, leave=False):
        images = batch["image"].to(device)
        targets = batch["targets"].to(device)
        target_mask = batch["target_mask"].to(device)

        outputs = model(images)

        # 여러 head의 점수를 한 줄로 모읍니다.
        # 비유하면 과목별 점수를 성적표 한 줄에 적는 과정입니다.
        preds = torch.stack([outputs["scores"][target] for target in TARGET_COLUMNS], dim=1)
        loss = masked_mse_loss(preds, targets, target_mask)

        if training:
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

        total_loss += loss.item()
        total_steps += 1

    return total_loss / max(total_steps, 1)


In [ ]:
for epoch in range(start_epoch, EPOCHS + 1):
    train_loss = run_epoch(model, train_loader, optimizer)
    with torch.no_grad():
        val_loss = run_epoch(model, val_loader, optimizer=None)

    print(f"[Epoch {epoch}] train_loss={train_loss:.4f} val_loss={val_loss:.4f}")

    save_checkpoint(
        path=str(OUTPUT_DIR / "last.pt"),
        model=model,
        optimizer=optimizer,
        epoch=epoch,
        metrics={"train_loss": train_loss, "val_loss": val_loss},
        extra_state={"target_columns": TARGET_COLUMNS, "backbone": "efficientnet_b3", "best_val_loss": best_val},
    )

    if val_loss < best_val:
        best_val = val_loss
        save_checkpoint(
            path=str(OUTPUT_DIR / "best.pt"),
            model=model,
            optimizer=optimizer,
            epoch=epoch,
            metrics={"train_loss": train_loss, "val_loss": val_loss},
            extra_state={"target_columns": TARGET_COLUMNS, "backbone": "efficientnet_b3", "best_val_loss": best_val},
        )


## 학습이 끝나면

- `best.pt`가 가장 성능이 좋았던 모델입니다.
- 이 모델은 사진만 보고 현재 피부 상태 점수를 예측합니다.
- 다음 노트북에서는 이 점수 흐름을 가지고 원인 분석을 합니다.
